# LESSON 6.6: Minimum Mean Square Error (Wiener) Filtering
## Image Restoration

In this lesson:
- The Wiener filter derivation and formula
- Understanding the noise-to-signal power ratio
- Effect of the Wiener parameter K
- Comparison with inverse filtering
- Wiener filter for atmospheric turbulence restoration
- Wiener filter for motion blur restoration
- Parametric Wiener filtering and its practical advantages

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## 1. Motivation: Why Wiener Filtering?

We saw that **inverse filtering** fails because it amplifies noise:

$$\hat{F}_{\text{inverse}} = \frac{G}{H} = F + \frac{N}{H}$$

Where $H \to 0$, the noise term $N/H \to \infty$.

**Question**: Can we design a filter that **optimally balances** deblurring and noise suppression?

**Answer**: Yes! The **Wiener filter** (Norbert Wiener, 1949) minimizes the **Mean Square Error** between the estimate $\hat{f}$ and the true image $f$:

$$\min E\left\{|f(x,y) - \hat{f}(x,y)|^2\right\}$$

The Wiener filter is the **optimal linear filter** in the MSE sense.

---
## 2. The Wiener Filter Formula

### Full Wiener Filter:

$$\boxed{\hat{F}(u,v) = \left[\frac{H^*(u,v)}{|H(u,v)|^2 + S_\eta(u,v)/S_f(u,v)}\right] G(u,v)}$$

Which can also be written as:

$$\hat{F}(u,v) = \left[\frac{1}{H(u,v)} \cdot \frac{|H(u,v)|^2}{|H(u,v)|^2 + S_\eta/S_f}\right] G(u,v)$$

Where:
- $H^*(u,v)$ = complex conjugate of $H$
- $|H(u,v)|^2 = H \cdot H^*$ = power of the degradation
- $S_\eta(u,v) = |N(u,v)|^2$ = **noise power spectrum**
- $S_f(u,v) = |F(u,v)|^2$ = **signal power spectrum**
- $S_\eta / S_f$ = **noise-to-signal power ratio**

### Interpretation:

The Wiener filter is the **inverse filter** multiplied by a **damping factor**:

$$W(u,v) = \underbrace{\frac{1}{H(u,v)}}_{\text{Inverse filter}} \cdot \underbrace{\frac{|H(u,v)|^2}{|H(u,v)|^2 + S_\eta/S_f}}_{\text{Damping factor}}$$

### Behavior:
- When $S_\eta / S_f \to 0$ (no noise): damping → 1, Wiener → inverse filter
- When $S_\eta / S_f \to \infty$ (all noise): damping → 0, output suppressed
- At each frequency, the filter **automatically balances** between deblurring and noise suppression

In [ ]:
# Helper functions

def create_test_image(size=256):
    """Create a synthetic biomedical phantom image."""
    img = np.ones((size, size), dtype=np.float64) * 30
    Y, X = np.mgrid[0:size, 0:size]
    cx, cy = size // 2, size // 2
    
    body = ((X - cx) / 100) ** 2 + ((Y - cy) / 80) ** 2 <= 1
    img[body] = 120
    organ1 = ((X - cx + 30) / 35) ** 2 + ((Y - cy + 10) / 45) ** 2 <= 1
    img[organ1] = 170
    organ2 = ((X - cx - 35) / 25) ** 2 + ((Y - cy - 15) / 30) ** 2 <= 1
    img[organ2] = 80
    for (sx, sy, sr) in [(cx-20, cy+30, 5), (cx+40, cy-25, 4), (cx-50, cy-20, 3)]:
        spot = (X - sx) ** 2 + (Y - sy) ** 2 <= sr ** 2
        img[spot] = 240
    for i in range(cy-40, cy+40):
        j = int(cx + 20 * np.sin(2 * np.pi * i / 50))
        if 0 <= j < size and 0 <= i < size:
            img[i, max(0,j-1):min(size,j+2)] = 200
    return img


def compute_psnr(original, restored):
    """Compute PSNR in dB."""
    mse = np.mean((original - restored) ** 2)
    if mse == 0:
        return float('inf')
    return 10 * np.log10(255.0 ** 2 / mse)


def atmospheric_turbulence_otf(P, Q, k=0.0025):
    """Atmospheric turbulence degradation OTF."""
    u = np.arange(P) - P/2
    v = np.arange(Q) - Q/2
    U, V = np.meshgrid(u, v, indexing='ij')
    D2 = U**2 + V**2
    H = np.exp(-k * D2**(5/6))
    return H


def motion_blur_otf(P, Q, a=0.1, b=0.0, T=1.0):
    """Uniform linear motion blur OTF."""
    u = np.arange(P) - P/2
    v = np.arange(Q) - Q/2
    U, V = np.meshgrid(u, v, indexing='ij')
    arg = np.pi * (U * a + V * b)
    arg_safe = np.where(np.abs(arg) < 1e-10, 1e-10, arg)
    H = (T / arg_safe) * np.sin(arg_safe) * np.exp(-1j * arg_safe)
    H[np.abs(arg) < 1e-10] = T
    return H


original = create_test_image(256)
M, N = original.shape
print(f"Test image: {M}×{N}")

In [ ]:
# Implement the Wiener filter

def wiener_filter(G, H, K):
    """
    Parametric Wiener filter.
    
    F_hat = (H* / (|H|^2 + K)) * G
    
    Parameters:
        G: DFT of degraded image (centered)
        H: degradation OTF (centered)
        K: noise-to-signal ratio (scalar constant)
           K = S_eta / S_f (can be approximated as a constant)
    Returns:
        Restored image
    """
    H_conj = np.conj(H)
    H_mag2 = np.abs(H) ** 2
    
    # Wiener filter transfer function
    W = H_conj / (H_mag2 + K)
    
    F_hat = W * G
    restored = np.real(np.fft.ifft2(np.fft.ifftshift(F_hat)))
    return np.clip(restored, 0, 255)


def inverse_filter(G, H, epsilon=1e-3):
    """
    Pseudoinverse filter for comparison.
    """
    H_safe = np.where(np.abs(H) > epsilon, H, epsilon * np.exp(1j * np.angle(H)))
    F_hat = G / H_safe
    restored = np.real(np.fft.ifft2(np.fft.ifftshift(F_hat)))
    return np.clip(restored, 0, 255)


print("Wiener filter function defined.")

---
## 3. Parametric Wiener Filter

In practice, the power spectra $S_\eta$ and $S_f$ are **unknown**. A common simplification is to use a **constant** $K$:

$$\hat{F}(u,v) = \frac{H^*(u,v)}{|H(u,v)|^2 + K} \cdot G(u,v)$$

This is called the **parametric Wiener filter** (Gonzalez, Section 5.8).

### Effect of $K$:
- $K = 0$: Wiener reduces to inverse filter (no noise regularization)
- $K$ small: aggressive deblurring, more noise amplification
- $K$ large: strong noise suppression, less deblurring (over-smoothed)
- **Optimal $K$**: best balance (gives maximum PSNR)

### Relationship to noise:
If the noise is white with variance $\sigma_\eta^2$ and the image power spectrum is approximately $S_f$:

$$K \approx \frac{\sigma_\eta^2 \cdot M \cdot N}{\sum |F(u,v)|^2}$$

In [ ]:
# Demonstrate the effect of K on Wiener filtering

# Degrade with atmospheric turbulence + noise
k_turb = 0.005
H_turb = atmospheric_turbulence_otf(M, N, k_turb)
F_orig = np.fft.fftshift(np.fft.fft2(original))

np.random.seed(42)
noise_sigma = 3
degraded = np.real(np.fft.ifft2(np.fft.ifftshift(H_turb * F_orig)))
degraded += np.random.normal(0, noise_sigma, degraded.shape)
degraded = np.clip(degraded, 0, 255)

G = np.fft.fftshift(np.fft.fft2(degraded))

# Apply Wiener filter with different K values
K_values = [0.0001, 0.001, 0.005, 0.01, 0.05, 0.1]

fig, axes = plt.subplots(2, 6, figsize=(22, 7))

for i, K in enumerate(K_values):
    restored = wiener_filter(G, H_turb, K)
    psnr = compute_psnr(original, restored)
    
    # Show the Wiener filter transfer function
    W = np.abs(np.conj(H_turb) / (np.abs(H_turb)**2 + K))
    axes[0, i].imshow(W, cmap='gray')
    axes[0, i].set_title(f'|W(u,v)| K={K}', fontsize=10)
    axes[0, i].axis('off')
    
    axes[1, i].imshow(restored, cmap='gray', vmin=0, vmax=255)
    axes[1, i].set_title(f'K={K}\nPSNR={psnr:.1f} dB', fontsize=10)
    axes[1, i].axis('off')

axes[0, 0].set_ylabel('Wiener TF', fontsize=12)
axes[1, 0].set_ylabel('Restored', fontsize=12)

plt.suptitle('Effect of Wiener Parameter K on Restoration Quality\n'
             'Small K = noisy | Optimal K = balanced | Large K = over-smoothed',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Find the optimal K value

K_range = np.logspace(-5, 0, 200)
psnr_values = []

for K in K_range:
    restored = wiener_filter(G, H_turb, K)
    psnr_values.append(compute_psnr(original, restored))

best_idx = np.argmax(psnr_values)
best_K = K_range[best_idx]
best_psnr = psnr_values[best_idx]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].semilogx(K_range, psnr_values, 'b-', linewidth=2)
axes[0].axvline(x=best_K, color='r', linestyle='--', label=f'Best K={best_K:.5f}')
axes[0].set_xlabel('K (log scale)', fontsize=12)
axes[0].set_ylabel('PSNR (dB)', fontsize=12)
axes[0].set_title('PSNR vs Wiener Parameter K', fontsize=13)
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

axes[1].imshow(degraded, cmap='gray', vmin=0, vmax=255)
axes[1].set_title(f'Degraded\nPSNR={compute_psnr(original, degraded):.1f} dB', fontsize=12)
axes[1].axis('off')

best_restored = wiener_filter(G, H_turb, best_K)
axes[2].imshow(best_restored, cmap='gray', vmin=0, vmax=255)
axes[2].set_title(f'Optimal Wiener\nK={best_K:.5f}, PSNR={best_psnr:.1f} dB', fontsize=12)
axes[2].axis('off')

plt.suptitle('Finding the Optimal Wiener Parameter K',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"Degraded PSNR: {compute_psnr(original, degraded):.1f} dB")
print(f"Optimal K = {best_K:.5f}, Restored PSNR = {best_psnr:.1f} dB")
print(f"Improvement: +{best_psnr - compute_psnr(original, degraded):.1f} dB")

---
## 4. Wiener Filter vs Inverse Filter

The key advantage of Wiener filtering over inverse filtering is its behavior near the **zeros of H**:

| Property | Inverse Filter | Wiener Filter |
|---|---|---|
| Formula | $\hat{F} = G / H$ | $\hat{F} = \frac{H^*}{|H|^2 + K} G$ |
| When $H \to 0$ | $\hat{F} \to \infty$ (noise explosion) | $\hat{F} \to 0$ (safely suppressed) |
| Noise handling | None | Optimal MSE | 
| Parameters | None (or $\epsilon$) | $K$ (noise/signal ratio) |
| Result | Works only with zero noise | Works with any noise level |

In [ ]:
# Direct comparison: Inverse vs Wiener at different noise levels

noise_sigmas = [0.5, 2, 5, 15]

fig, axes = plt.subplots(3, 4, figsize=(18, 13))

for i, ns in enumerate(noise_sigmas):
    # Create degraded image
    np.random.seed(42)
    deg = np.real(np.fft.ifft2(np.fft.ifftshift(H_turb * F_orig)))
    deg += np.random.normal(0, ns, deg.shape)
    deg = np.clip(deg, 0, 255)
    
    G_i = np.fft.fftshift(np.fft.fft2(deg))
    
    # Inverse filter
    restored_inv = inverse_filter(G_i, H_turb, epsilon=0.01)
    
    # Wiener filter (find best K)
    best_psnr_w = -np.inf
    best_K_w = 0.01
    for K_try in np.logspace(-5, 0, 100):
        res = wiener_filter(G_i, H_turb, K_try)
        p = compute_psnr(original, res)
        if p > best_psnr_w:
            best_psnr_w = p
            best_K_w = K_try
    restored_wien = wiener_filter(G_i, H_turb, best_K_w)
    
    axes[0, i].imshow(deg, cmap='gray', vmin=0, vmax=255)
    psnr_deg = compute_psnr(original, deg)
    axes[0, i].set_title(f'Noise σ={ns}\nPSNR={psnr_deg:.1f} dB', fontsize=11)
    axes[0, i].axis('off')
    
    axes[1, i].imshow(restored_inv, cmap='gray', vmin=0, vmax=255)
    psnr_inv = compute_psnr(original, restored_inv)
    axes[1, i].set_title(f'Inverse\nPSNR={psnr_inv:.1f} dB', fontsize=11)
    axes[1, i].axis('off')
    
    axes[2, i].imshow(restored_wien, cmap='gray', vmin=0, vmax=255)
    axes[2, i].set_title(f'Wiener (K={best_K_w:.4f})\nPSNR={best_psnr_w:.1f} dB', fontsize=11)
    axes[2, i].axis('off')

axes[0, 0].set_ylabel('Degraded', fontsize=12)
axes[1, 0].set_ylabel('Inverse', fontsize=12)
axes[2, 0].set_ylabel('Wiener', fontsize=12)

plt.suptitle('Inverse Filter vs Wiener Filter at Different Noise Levels\n'
             'Wiener consistently outperforms inverse filtering',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Visualize HOW the Wiener filter handles the H→0 problem

# Use motion blur (has exact zeros)
H_motion = motion_blur_otf(M, N, a=0.05, b=0.0)
H_mag = np.abs(H_motion)
center = M // 2

K = 0.01
H_mag2 = H_mag ** 2

# 1/H (inverse filter) — diverges at zeros
inv_profile = 1.0 / np.maximum(H_mag[center, :], 1e-5)

# Wiener filter profile — stays bounded
wien_profile = H_mag[center, :] / (H_mag2[center, :] + K)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(H_mag[center, :], 'b-', linewidth=2)
axes[0].set_title('|H(u,v)| — Motion Blur', fontsize=12)
axes[0].set_xlabel('Frequency')
axes[0].set_ylabel('|H|')
axes[0].grid(True, alpha=0.3)

axes[1].plot(np.minimum(inv_profile, 200), 'r-', linewidth=2)
axes[1].set_title('|1/H| — Inverse Filter\n(diverges at zeros!)', fontsize=12)
axes[1].set_xlabel('Frequency')
axes[1].set_ylabel('|1/H|')
axes[1].grid(True, alpha=0.3)

axes[2].plot(wien_profile, 'g-', linewidth=2)
axes[2].set_title(f'|W| — Wiener Filter (K={K})\n(stays bounded everywhere!)', fontsize=12)
axes[2].set_xlabel('Frequency')
axes[2].set_ylabel('|W|')
axes[2].grid(True, alpha=0.3)

plt.suptitle('Why Wiener Works: It Stays Bounded Where H→0',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("Inverse filter: 1/H → ∞ at zeros → noise explosion")
print("Wiener filter: H*/(|H|²+K) → 0 at zeros → noise suppressed")
print("The parameter K acts as a regularization that prevents the filter from blowing up.")

In [ ]:
# Wiener filter for motion blur restoration

# Create motion-blurred noisy image
H_motion = motion_blur_otf(M, N, a=0.05, b=0.0)
F_orig = np.fft.fftshift(np.fft.fft2(original))

np.random.seed(42)
degraded_motion = np.real(np.fft.ifft2(np.fft.ifftshift(H_motion * F_orig)))
degraded_motion += np.random.normal(0, 3, degraded_motion.shape)
degraded_motion = np.clip(degraded_motion, 0, 255)

G_motion = np.fft.fftshift(np.fft.fft2(degraded_motion))

# Find optimal K
K_range = np.logspace(-5, 0, 200)
psnr_vals = [compute_psnr(original, wiener_filter(G_motion, H_motion, K)) for K in K_range]
best_K = K_range[np.argmax(psnr_vals)]

restored_motion = wiener_filter(G_motion, H_motion, best_K)
restored_inv_motion = inverse_filter(G_motion, H_motion, epsilon=0.01)

fig, axes = plt.subplots(2, 3, figsize=(15, 10))

axes[0, 0].imshow(original, cmap='gray', vmin=0, vmax=255)
axes[0, 0].set_title('Original', fontsize=12)
axes[0, 0].axis('off')

axes[0, 1].imshow(degraded_motion, cmap='gray', vmin=0, vmax=255)
axes[0, 1].set_title(f'Motion Blur + Noise\nPSNR={compute_psnr(original, degraded_motion):.1f} dB', fontsize=12)
axes[0, 1].axis('off')

axes[0, 2].imshow(np.abs(H_motion), cmap='gray')
axes[0, 2].set_title('|H(u,v)| Motion Blur OTF', fontsize=12)
axes[0, 2].axis('off')

axes[1, 0].imshow(restored_inv_motion, cmap='gray', vmin=0, vmax=255)
axes[1, 0].set_title(f'Inverse Filter\nPSNR={compute_psnr(original, restored_inv_motion):.1f} dB', fontsize=12)
axes[1, 0].axis('off')

axes[1, 1].imshow(restored_motion, cmap='gray', vmin=0, vmax=255)
axes[1, 1].set_title(f'Wiener Filter\nK={best_K:.5f}, PSNR={compute_psnr(original, restored_motion):.1f} dB', fontsize=12)
axes[1, 1].axis('off')

# Profile comparison
row = 128
axes[1, 2].plot(original[row, :], 'k-', linewidth=1, alpha=0.5, label='Original')
axes[1, 2].plot(degraded_motion[row, :], 'r-', linewidth=1, alpha=0.5, label='Degraded')
axes[1, 2].plot(restored_motion[row, :], 'b-', linewidth=1.5, label='Wiener')
axes[1, 2].set_title(f'Row {row} Profile', fontsize=12)
axes[1, 2].legend(fontsize=10)
axes[1, 2].grid(True, alpha=0.3)
axes[1, 2].set_xlabel('Column')
axes[1, 2].set_ylabel('Intensity')

plt.suptitle('Motion Blur Restoration: Inverse vs Wiener Filter',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Wiener filter across different degradation severities

k_values = [0.001, 0.003, 0.005, 0.01]
noise_sigma = 3

fig, axes = plt.subplots(3, 4, figsize=(18, 13))

for i, k in enumerate(k_values):
    H = atmospheric_turbulence_otf(M, N, k)
    F_orig = np.fft.fftshift(np.fft.fft2(original))
    
    np.random.seed(42)
    deg = np.real(np.fft.ifft2(np.fft.ifftshift(H * F_orig)))
    deg += np.random.normal(0, noise_sigma, deg.shape)
    deg = np.clip(deg, 0, 255)
    
    G = np.fft.fftshift(np.fft.fft2(deg))
    
    # Find optimal K
    K_search = np.logspace(-5, 0, 100)
    psnrs = [compute_psnr(original, wiener_filter(G, H, K_try)) for K_try in K_search]
    best_K = K_search[np.argmax(psnrs)]
    restored = wiener_filter(G, H, best_K)
    
    axes[0, i].imshow(H, cmap='gray', vmin=0, vmax=1)
    axes[0, i].set_title(f'H(u,v): k={k}', fontsize=11)
    axes[0, i].axis('off')
    
    axes[1, i].imshow(deg, cmap='gray', vmin=0, vmax=255)
    axes[1, i].set_title(f'Degraded\nPSNR={compute_psnr(original, deg):.1f}', fontsize=11)
    axes[1, i].axis('off')
    
    axes[2, i].imshow(restored, cmap='gray', vmin=0, vmax=255)
    axes[2, i].set_title(f'Wiener\nPSNR={compute_psnr(original, restored):.1f}', fontsize=11)
    axes[2, i].axis('off')

axes[0, 0].set_ylabel('OTF', fontsize=12)
axes[1, 0].set_ylabel('Degraded', fontsize=12)
axes[2, 0].set_ylabel('Wiener', fontsize=12)

plt.suptitle('Wiener Filter Across Different Turbulence Severities\n'
             'Noise σ=3, K automatically optimized for each case',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("The Wiener filter consistently improves image quality across all degradation levels.")
print("More severe degradation requires a larger K (more regularization).")

---
## 5. Practical Considerations

### How to choose K in practice?

Since the original image $f$ is unknown, we cannot compute the optimal $K$ directly. Several approaches:

1. **Interactive adjustment**: try different $K$ values and visually evaluate
2. **Noise estimation**: estimate $\sigma_\eta^2$ from a flat region, then $K \approx \sigma_\eta^2 \cdot M \cdot N / \text{total\_signal\_power}$
3. **Generalized Cross-Validation (GCV)**: automatic parameter selection without knowing the original

### Limitations of Wiener Filter:
- Assumes **stationary** noise and signal (statistics don't change across the image)
- Assumes the degradation $H$ is **exactly known**
- The constant-K approximation may not be optimal for all spatial frequencies
- For non-stationary degradation, **adaptive** methods are needed

In [ ]:
# Practical K estimation from noise variance

# Known noise sigma
true_noise_sigma = 5

# Degrade
H = atmospheric_turbulence_otf(M, N, k=0.005)
F_orig = np.fft.fftshift(np.fft.fft2(original))

np.random.seed(42)
degraded = np.real(np.fft.ifft2(np.fft.ifftshift(H * F_orig)))
degraded += np.random.normal(0, true_noise_sigma, degraded.shape)
degraded = np.clip(degraded, 0, 255)

G = np.fft.fftshift(np.fft.fft2(degraded))

# Estimate K from noise variance
noise_power = true_noise_sigma**2 * M * N  # total noise power
signal_power = np.sum(np.abs(F_orig)**2)   # total signal power
K_estimated = noise_power / signal_power

# Find true optimal K for comparison
K_range = np.logspace(-5, 0, 200)
psnrs = [compute_psnr(original, wiener_filter(G, H, K)) for K in K_range]
K_optimal = K_range[np.argmax(psnrs)]

# Compare
restored_est = wiener_filter(G, H, K_estimated)
restored_opt = wiener_filter(G, H, K_optimal)

fig, axes = plt.subplots(1, 4, figsize=(18, 5))

axes[0].imshow(original, cmap='gray', vmin=0, vmax=255)
axes[0].set_title('Original', fontsize=12)
axes[0].axis('off')

axes[1].imshow(degraded, cmap='gray', vmin=0, vmax=255)
axes[1].set_title(f'Degraded\nPSNR={compute_psnr(original, degraded):.1f} dB', fontsize=12)
axes[1].axis('off')

axes[2].imshow(restored_est, cmap='gray', vmin=0, vmax=255)
axes[2].set_title(f'Estimated K={K_estimated:.5f}\nPSNR={compute_psnr(original, restored_est):.1f} dB', fontsize=12)
axes[2].axis('off')

axes[3].imshow(restored_opt, cmap='gray', vmin=0, vmax=255)
axes[3].set_title(f'Optimal K={K_optimal:.5f}\nPSNR={compute_psnr(original, restored_opt):.1f} dB', fontsize=12)
axes[3].axis('off')

plt.suptitle('Estimated vs Optimal K: The Estimate is Often Close Enough',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"K estimated from noise variance: {K_estimated:.5f}")
print(f"K optimal (using original):      {K_optimal:.5f}")
print(f"PSNR with estimated K: {compute_psnr(original, restored_est):.1f} dB")
print(f"PSNR with optimal K:   {compute_psnr(original, restored_opt):.1f} dB")

---
## Summary

What we learned:

1. **Wiener filter** minimizes the Mean Square Error $E\{|f - \hat{f}|^2\}$ and is the optimal linear restoration filter.

2. **Formula**: $\hat{F} = \frac{H^*}{|H|^2 + K} \cdot G$, where $K$ approximates the noise-to-signal ratio $S_\eta / S_f$.

3. **Key advantage over inverse filter**: where $H \to 0$, the Wiener filter output → 0 (safe), while inverse filter → ∞ (catastrophic).

4. **Parameter $K$** controls the trade-off: $K = 0$ → inverse filter, $K$ large → over-smoothing. The optimal $K$ gives the best PSNR.

5. **In practice**, $K$ can be estimated from the noise variance: $K \approx \sigma_\eta^2 \cdot MN / \sum|F|^2$.

6. **Wiener filter consistently outperforms** inverse filtering, especially at higher noise levels and for degradations with zeros (like motion blur).

7. **Limitation**: Wiener filter assumes stationary statistics and exact knowledge of $H$. The Constrained Least Squares filter (next lesson) requires less prior knowledge.